# Assignment 2: Imaging Pipeline

MIDS W281: Computer Vision

## Recommended Libraries

In [1]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import os
from glob import glob

## Part 1: Demosaicing

![Bayer Pattern](https://raw.githubusercontent.com/W281/fileRepository/main/Assignments/Assignment_2/bayer_pattern.png)

### Overview
In this exercise you will convert a raw sensor image into a full color image using demosaicing. Digital sensors record color images through a Bayer mosaic (above), where each pixel records only one of the three color colors (RGB). A software interpolation is then needed to reconstruct all three colors at each pixel.

**HINT: There are different Bayer mosaic patterns than the one shown above, so you should look into them if you want the correct setup for Part 1**

### Description: 
We will provide you with some raw images, represented as grayscale images (red, green, and blue pixels are all on the same channel of the image). Your task is to write some python code to demosaic and generate a full three-channel RGB image. You're encouraged to debug your code using the image `signs-small.png` because it is not very large and exhibits some interesting challenges of demosaicing. Note that you may need to convert images between `UINT8` and `float32` data types for computation and visualization.

For simplicity we will ignore the pixels at the boundary of the image, specifically the first and last two rows and columns don't need to be reconstructed. This will allow you to focus on the general case and not worry about whether neighboring values are unavailable. It's actually not uncommon for cameras and software to return a slightly-cropped image for similar reasons. Therefore, for an image of size NxN, you will return a cropped image of size (N-2)x(N-2)x3.  

1. Write a python function that takes as input a raw image and offset and returns a single-channel 2-D image corresponding to the interpolated green channel. The offset encodes whether either the top-left pixel or its right neighbor is the first green pixel. In our Figure 1 example, the second pixel is green, so offset=1.  

2. Write another python function for generating the red and blue channels. This function takes a raw image and two offsets: one for row offset and one for column offset, and returns a single-channel image. The row/column offset for the red channel is (0,0) and for the blue channel (1,1) for the Figure 1 example, but that might not be the case for the signs images (see hint). Note that the interpolation for the red/blue channel will be different than the green channel because the recorded pixels are sparser. For interpolated pixels that have two direct neighbors that are known (left-right or up-down), simply take the linear average between the two values. For the remaining case, average the four diagonal pixels. You can ignore the first and last two rows or columns to make sure that you have all the neighbors you need. Similar to the green-channel, interpolate the values when they are missing and copy the values when they are available.  

3. Using the above two functions, create a full three-channel, RGB image. You might observe some checkerboard artifacts around strong edges. This is expected from our naive interpolation approach.

### Deliverables:
- Python code for interpolation of green channel and red/blue channel that handles different Bayer mosaic patterns dictated by the offsets
- Full-three channel RGB image for `signs.png` (**THE BIG ONE, NOT THE SMALL ONE**)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def _to_float(img):
    img = np.asarray(img)
    if img.ndim == 3:
        img = img[..., 0]
    img = img.astype(np.float32)
    if img.max() > 1.5:
        img = img / (65535.0 if img.max() > 255 else 255.0)
    return img

def interpolate_green(raw_img, offset_value):
    raw = _to_float(raw_img)
    h, w = raw.shape
    rr, cc = np.indices((h, w))
    is_green = ((rr + cc) % 2 == offset_value)
    p = np.pad(raw, 1, mode="edge")
    interp = 0.25 * (p[:-2, 1:-1] + p[2:, 1:-1] + p[1:-1, :-2] + p[1:-1, 2:])
    green = np.where(is_green, raw, interp)
    return green[2:-2, 2:-2]

def interpolate_redblue(raw_img, offset_pair):
    raw = _to_float(raw_img)
    h, w = raw.shape
    r0, c0 = offset_pair
    rr, cc = np.indices((h, w))
    known = (rr % 2 == r0) & (cc % 2 == c0)
    horiz_known = (rr % 2 == r0) & (cc % 2 != c0)
    vert_known = (rr % 2 != r0) & (cc % 2 == c0)
    p = np.pad(raw, 1, mode="edge")
    h_avg = 0.5 * (p[1:-1, :-2] + p[1:-1, 2:])
    v_avg = 0.5 * (p[:-2, 1:-1] + p[2:, 1:-1])
    d_avg = 0.25 * (p[:-2, :-2] + p[:-2, 2:] + p[2:, :-2] + p[2:, 2:])
    out = np.where(known, raw, np.where(horiz_known, h_avg, np.where(vert_known, v_avg, d_avg)))
    return out[2:-2, 2:-2]

test_img = plt.imread("./demosaicing/signs-small.png")
raw_img = plt.imread("./demosaicing/signs.png")
print("small", np.asarray(test_img).dtype, np.asarray(test_img).shape)
print("signs.png", np.asarray(raw_img).dtype, np.asarray(raw_img).shape)

# signs.png is GRBG (green at top-left). Figure 1 RGGB does not match this capture.
green_offset = 0
red_offset = (0, 1)
blue_offset = (1, 0)

green_img = interpolate_green(raw_img, green_offset)
red_img = interpolate_redblue(raw_img, red_offset)
blue_img = interpolate_redblue(raw_img, blue_offset)
color_img = np.clip(np.stack([red_img, green_img, blue_img], axis=-1), 0, 1)

plt.figure(figsize=(12, 8))
plt.imshow(color_img)
plt.axis("off")
plt.title("Demosaiced signs.png")
plt.show()


## Part 2: Denoising

![Denoising Teaser](https://raw.githubusercontent.com/W281/fileRepository/main/Assignments/Assignment_2/denoising.png)

### Overview
Random noise is a problem that often arises in cameras specially in extremely low light conditions, and its presence can seriously degrade the quality of a digital image. To remedy the situation, an average of multiple images, captured very close in time, can be used to improve the quality final image. Because the camera may move while recording mulitple images, we will need to align the images before averaging and denoising. We will implement this alignment + denoising algorithm.

### Description: 
We provide 18 images captured in a low light setting. One of the images is shown above.

Each image is slightly mis-aligned from the previous image in the sequence. Our goal is to align each of the images in the sequence to the first image and then average the aligned images to reduce the noise (which tends to be independent across images). Note that you may need to convert images between `UINT8` and `float32` data types for computation and visualization.

1. Write a python function that takes as input two images and returns the horizontal and vertical offset that best aligns the two images. Ignore the difference for all the pixels less than or equal to a `maxOffset` away from the edges. Use a brute force approach that tries every possible integer translation and evaluates the quality of a match using the squared error norm (the sum of the squared pixel differences). You can set the `maxOffset` to 15 pixels.  
2. Align each image to the first image and denoise by averaging all of the aligned images.

**HINT: sweeping through all of the offsets means more than just going right and down**

### Deliverables:

- Python code to align noisy images
- Aligned and de-noised average image

In [ ]:
from glob import glob
import numpy as np
import matplotlib.pyplot as plt

img_list = sorted(glob("./denoising/*.png"))
imgs = []
for p in img_list:
    im = plt.imread(p)
    if im.ndim == 3:
        im = im[..., :3]
    im = im.astype(np.float32)
    if im.max() > 1.5:
        im = im / 255.0
    imgs.append(im)
    print(p, im.shape, im.dtype, im.min(), im.max())

img_ref = imgs[0]
maxOffset = 15

def align_imgs(img1, img2, maxOffset):
    a = img1.astype(np.float32)
    b = img2.astype(np.float32)
    h, w = a.shape[:2]
    m = maxOffset
    ref = a[m:-m, m:-m]
    best_error = np.inf
    best_offset = (0, 0)
    for dy in range(-m, m + 1):
        for dx in range(-m, m + 1):
            moved = b[m + dy : h - m + dy, m + dx : w - m + dx]
            err = np.sum((ref - moved) ** 2)
            if err < best_error:
                best_error = err
                best_offset = (dx, dy)
    return best_offset, best_error

def combine_imgs(img_list, offset_list):
    acc = np.zeros_like(img_list[0], dtype=np.float32)
    n = len(img_list)
    for im, (dx, dy) in zip(img_list, offset_list):
        acc += np.roll(np.roll(im, dy, axis=0), dx, axis=1) / n
    return np.clip(acc, 0, 1)

offset_list = []
for i, im in enumerate(imgs):
    if i == 0:
        offset_list.append((0, 0))
        continue
    off, err = align_imgs(img_ref, im, maxOffset)
    offset_list.append(off)
    print(i, off, err)

composite_img = combine_imgs(imgs, offset_list)

plt.figure(figsize=(20, 10))
plt.axis("off")
plt.imshow(composite_img)
plt.title("Aligned average (denoised)")
plt.show()


## Part 3: White balance

Before WB

![WB Teaser](https://raw.githubusercontent.com/W281/fileRepository/main/Assignments/Assignment_2/white_balance/input.png)

After WB Gray-world

![WB Teaser](https://raw.githubusercontent.com/W281/fileRepository/main/Assignments/Assignment_2/white_balance/output.png)

After WB White-patch

![WB Teaser](https://raw.githubusercontent.com/W281/fileRepository/main/Assignments/Assignment_2/white_balance/output_wp.png)

### Overview
Color constancy is one of the most amazing features of the human visual system. When we look at objects under different illuminations, their colors stay relatively constant. This helps humans to more easily identify objects under varying illuminations. A similar behavior is highly desirable in digital still and video cameras. This is achieved via white balancing, typically employed in a digital camera's imaging pipeline to adjust the coloration of images captured under different illuminations.

### Description 
You will implement two methods for white balancing, gray-world and white-patch, described below. The two images above show before and after white balancing using the gray-world assumption. You can test your white balancing code using this example image. Images which are white-balanced using the white-patch method will have a different appearance depending on the patch selected.

1. One simple technique for white balancing is based on the gray-world assumption. This assumption argues that the average reflectance of a scene is achromatic. In other words, the mean of the red, green, and blue channels in a given scene should be roughly equal. We will implement this white balancing technique. Write a function to automatically white balance an image using the gray-world assumption. You should multiply each color channel by a scale factor so that the resulting mean of each of the three color channels is the same and equal to the average value of the green channel of the input image.

2. Another method for white balancing uses a white-patch in the image. In this method, the user manually selects an image region which is supposed to be white but looks colored due to the scene illumination. As above, we will scale each color channel by a factor so that the average color of the selected region becomes white. Write a python code to implement the white-patch balancing method. Your code should take in a location in the image and use a fixed-sized region around that point to compute the target white point.  You will know your output is correct if things that are supposed to be white look more white (e.g. clouds).

In both cases, you will need to account for pixel values that fall outside the displayable range after transformation. You should clip these values rather than scaling them.

Please specify the region that you selected in your code for 2.

### Deliverables:

- Python functions for gray-world and white-patch white balancing
- Output images after white balancing the image `white_balance/input.png` using both gray-world and white-patch methods

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

input_img = plt.imread("./white_balance/input.png")
print("input dtype/range", input_img.dtype, input_img.min(), input_img.max())
img = input_img.astype(np.float32)
if img.max() > 1.5:
    img = img / 255.0
img = img[..., :3]

def gray_world(input_img):
    x = input_img.copy().astype(np.float32)
    mean_r, mean_g, mean_b = x[..., 0].mean(), x[..., 1].mean(), x[..., 2].mean()
    x[..., 0] *= mean_g / mean_r
    x[..., 2] *= mean_g / mean_b
    return np.clip(x, 0, 1)

def white_patch(input_img, center=(420, 80), half=20):
    """Fixed patch on the bright clouds near the top of the frame.
    center is (x, y) in image coordinates; region is (2*half+1)^2.
    """
    x = input_img.copy().astype(np.float32)
    cx, cy = center
    y0, y1 = max(0, cy - half), min(x.shape[0], cy + half + 1)
    x0, x1 = max(0, cx - half), min(x.shape[1], cx + half + 1)
    patch = x[y0:y1, x0:x1]
    means = patch.reshape(-1, 3).mean(axis=0)
    # scale so the patch mean becomes white (1,1,1)
    x[..., 0] *= 1.0 / means[0]
    x[..., 1] *= 1.0 / means[1]
    x[..., 2] *= 1.0 / means[2]
    return np.clip(x, 0, 1)

gray_world_output = gray_world(img)
white_patch_output = white_patch(img, center=(420, 80), half=20)

fig1 = plt.figure()
plt.imshow(gray_world_output)
plt.xticks(ticks=[])
plt.yticks(ticks=[])
plt.title("Gray-World")
plt.show()

fig2 = plt.figure()
plt.imshow(white_patch_output)
plt.xticks(ticks=[])
plt.yticks(ticks=[])
plt.title("White-Patch (clouds around x=420, y=80)")
plt.show()


#### Acknowledgments
This assignment is based on an assignment for Computational Aspects of Digital Photography class by Prof. Wojciech Jarosz at Dartmouth College.